# Qwen3.5-4B frozen RotQuant K/V transfer

This focused follow-up reuses the completed K/V matrix recipes and candidate scores. It tests whether one fixed per-layer K/V bit map transfers between 256- and 1,024-token contexts without recalibration.

## Goal

Evaluate three fixed 3.25-bpv maps on both contexts: the short-context winner, the long-context winner, and a mixed-context map built from both selection splits. Select one universal deployment map only if it beats uniform K3/V3 at no more exact bytes in both held-out contexts. Otherwise retain two context buckets.

### Key assumptions

- The source matrix is the completed run pinned below; its JSON files remain in Google Drive.
- Recipes and mixed-map construction use only the original selection metrics. Final comparisons use the original disjoint evaluation calls.
- A replay gate must reproduce both context-optimized results before transfer metrics are trusted.
- The 4-bit RotQuant weight model is held fixed. This notebook isolates K/V-map transfer, not joint weight optimization.
- CUDA weight fallback is quality-only. K/V packed-byte accounting is exact logical storage.

## Setup

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/CodeHalwell/rotquant.git"
REPO_REF = "main"
REPO_DIR = Path("/content/rotquant-kv-transfer")
MODEL_ID = "unsloth/Qwen3.5-4B"
SOURCE_RUN_ID = "bf94f2045a90"
DRIVE_ROOT = Path("/content/drive/MyDrive/rotquant/qwen35_kv_matrix")
LOCAL_ROOT = Path("/content/qwen35_kv_frozen_transfer")
USE_GOOGLE_DRIVE = True
CONFIRM_FROZEN_TRANSFER = False
FORCE_RERUN = False
DOWNLOAD_RESULTS = True
REPLAY_KL_RTOL = 0.02
REPLAY_KL_ATOL = 1e-3
REPLAY_COSINE_ATOL = 0.005
REPLAY_TOP1_ATOL = 0.03125

print({"source_run": SOURCE_RUN_ID, "repo_ref": REPO_REF})

### 1. Verify CUDA

In [ ]:
import os
import subprocess
import sys
import torch

assert torch.cuda.is_available(), "Select a CUDA GPU runtime before continuing."
gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / 2**30
print(f"GPU: {gpu.name} | VRAM: {vram_gib:.1f} GiB | torch={torch.__version__} | CUDA={torch.version.cuda}")
assert vram_gib >= 40, "The cached fp16 weight fallback requires an approximately 48 GB or larger GPU."
subprocess.run(["nvidia-smi"], check=True)

### 2. Mount Drive, fetch `main`, and install the runtime

In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SOURCE_RESULT_ROOT = DRIVE_ROOT / SOURCE_RUN_ID
    RESULT_BASE = DRIVE_ROOT / "frozen_transfer"
else:
    SOURCE_RESULT_ROOT = LOCAL_ROOT / SOURCE_RUN_ID
    RESULT_BASE = LOCAL_ROOT / "results"
assert SOURCE_RESULT_ROOT.exists(), f"Missing completed matrix: {SOURCE_RESULT_ROOT}"
RESULT_BASE.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", REPO_REF], cwd=REPO_DIR, check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
RESULT_ROOT = RESULT_BASE / commit[:12]
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Using {commit}; source={SOURCE_RESULT_ROOT}; output={RESULT_ROOT}")

In [ ]:
runtime_packages = [
    "transformers>=5.9,<6", "datasets>=4.8", "accelerate",
    "safetensors", "sentencepiece", "scipy", "pyyaml",
    "pandas", "matplotlib", "huggingface_hub",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *runtime_packages], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "--no-deps"], check=True)
pil_probe_command = [sys.executable, "-c", "from PIL import Image, ImageColor, ImageDraw, ImageFont, ImageText; print(Image.__version__)"]
pil_probe = subprocess.run(pil_probe_command, capture_output=True, text=True)
if pil_probe.returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-cache-dir", "pillow==12.2.0"], check=True)
    subprocess.run(pil_probe_command, check=True)
    raise RuntimeError("Pillow was repaired. Restart the session and rerun from the top.")
os.environ["TORCH_ALLOW_TF32_CUBLAS_OVERRIDE"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")
print(f"Runtime ready; Pillow={pil_probe.stdout.strip()}")

## Data

### 3. Load and validate the completed matrix artifacts

In [ ]:
import json

def load_single(pattern):
    matches = sorted(SOURCE_RESULT_ROOT.glob(pattern))
    assert len(matches) == 1, f"Expected one {pattern}, found {matches}"
    with matches[0].open() as handle:
        return json.load(handle)

short_dynamic = load_single("dynamic_3.25bpv_s0_*.json")
long_dynamic = load_single("long_dynamic_3.25bpv_s0_*.json")
short_uniform3 = load_single("uniform_k3_v3_s0_*.json")
long_uniform3 = load_single("long_uniform_k3_v3_s0_*.json")
long_uniform4 = load_single("long_uniform_k4_v4_s0_*.json")

for payload in (short_dynamic, long_dynamic, short_uniform3, long_uniform3, long_uniform4):
    assert payload["git_sha"].startswith(SOURCE_RUN_ID), payload["git_sha"]
assert short_dynamic["metrics"]["dynamic"]["target_reached"]
assert long_dynamic["metrics"]["dynamic"]["target_reached"]
short_recipe = short_dynamic["metrics"]["dynamic"]["recipe"]
long_recipe = long_dynamic["metrics"]["dynamic"]["recipe"]
assert {row["layer"] for row in short_recipe} == {row["layer"] for row in long_recipe}
print({
    "short_source_kl": short_dynamic["metrics"]["mean_teacher_kl"],
    "long_source_kl": long_dynamic["metrics"]["mean_teacher_kl"],
    "kv_layers": len(short_recipe),
})

### 4. Build a mixed-context map from selection metrics only

In [ ]:
def candidate_index(stats):
    kl_relative_drift = abs(actual["mean_teacher_kl"] / expected["mean_teacher_kl"] - 1)
    cosine_absolute_drift = abs(actual["mean_logit_cosine"] - expected["mean_logit_cosine"])
    top1_absolute_drift = abs(actual["top1_agreement"] - expected["top1_agreement"])
    nll_absolute_drift = abs(actual["nll_delta"] - expected["nll_delta"])
    result = {
        (int(item["layer"]), item["side"]): {int(option["bits"]): option for option in item["options"]}
        for item in stats["candidate_scores"]
    }

def mixed_context_recipe(short_payload, long_payload):
    short_stats = short_payload["metrics"]["dynamic"]
    long_stats = long_payload["metrics"]["dynamic"]
    short_candidates = candidate_index(short_stats)
    long_candidates = candidate_index(long_stats)
    assert short_candidates.keys() == long_candidates.keys()
    for component in short_candidates:
        assert short_candidates[component].keys() == long_candidates[component].keys()

    highest = max(max(options) for options in short_candidates.values())
    high_bytes = next(int(row["packed_kv_bytes"]) for row in short_stats["uniform_trials"] if int(row["bits"]) == highest)
    short_scale = max(abs(next(float(row["score"]) for row in short_stats["uniform_trials"] if int(row["bits"]) == highest)), 1e-12)
    long_scale = max(abs(next(float(row["score"]) for row in long_stats["uniform_trials"] if int(row["bits"]) == highest)), 1e-12)
    ordered_bits = {component: sorted(options) for component, options in short_candidates.items()}
    selected = {component: len(bits) - 1 for component, bits in ordered_bits.items()}

    def stored_bytes():
        return high_bytes + sum(int(short_candidates[component][ordered_bits[component][index]]["bytes_delta"]) for component, index in selected.items())

    while stored_bytes() > int(short_stats["target_bytes"]):
        best = None
        for order, (component, index) in enumerate(selected.items()):
            if index == 0:
                continue
            current_bits = ordered_bits[component][index]
            lower_bits = ordered_bits[component][index - 1]
            current_short = short_candidates[component][current_bits]
            lower_short = short_candidates[component][lower_bits]
            current_long = long_candidates[component][current_bits]
            lower_long = long_candidates[component][lower_bits]
            savings = int(current_short["bytes_delta"]) - int(lower_short["bytes_delta"])
            if savings <= 0:
                continue
            short_penalty = max(float(lower_short["score"]) - float(current_short["score"]), 0.0) / short_scale
            long_penalty = max(float(lower_long["score"]) - float(current_long["score"]), 0.0) / long_scale
            key = ((short_penalty + long_penalty) / (2 * savings), -savings, order)
            if best is None or key < best[0]:
                best = (key, component)
        assert best is not None, "Mixed allocator could not reach its exact-byte target."
        selected[best[1]] -= 1

    by_layer = {}
    for (layer, side), index in selected.items():
        by_layer.setdefault(layer, {"layer": layer})[f"{side}_bits"] = ordered_bits[(layer, side)][index]
    recipe = [by_layer[layer] for layer in sorted(by_layer)]
    assert all({"layer", "key_bits", "value_bits"} <= set(row) for row in recipe)
    return recipe, {"estimated_short_bytes": stored_bytes(), "target_short_bytes": int(short_stats["target_bytes"])}

mixed_recipe, mixed_allocator = mixed_context_recipe(short_dynamic, long_dynamic)
print(json.dumps({"mixed_recipe": mixed_recipe, **mixed_allocator}, indent=2))

## Steps

### 5. Define the resumable frozen evaluator

In [ ]:
import gc
import hashlib
import time
from copy import deepcopy
from dataclasses import asdict

sys.path.insert(0, str(REPO_DIR))
from eval import kv_cache as kv_cache_module
assert Path(kv_cache_module.__file__).resolve().is_relative_to(REPO_DIR.resolve()), kv_cache_module.__file__
assert "frozen_recipe" in (REPO_DIR / "eval/kv_cache.py").read_text()
from eval.kv_cache import KVCacheEvalConfig, evaluate_kv_cache
from rotquant.patch import PatchConfig, patch_model
from rotquant.quantize import QuantConfig
from rotquant.utils import environment_record, set_seed
from scripts.run_experiment import build_calib_loader, footprint_metrics, load_hf_model

trial_records = {}
batch_cache = {}

def frozen_profile(context_payload, recipe):
    profile = deepcopy(context_payload["kv_config"])
    profile["dynamic"] = None
    profile["frozen_recipe"] = deepcopy(recipe)
    return profile

def batches_for(tokenizer, config):
    count = config.eval_offset_batches + config.batches
    seq_len = config.prompt_len + config.continuation_len + 1
    key = (count, seq_len, config.skip)
    if key not in batch_cache:
        batch_cache[key] = build_calib_loader(tokenizer, count, seq_len, "cuda", skip=config.skip)
    return batch_cache[key]

def trial_path(name, profile):
    signature = hashlib.sha256(json.dumps({"commit": commit, "source_run": SOURCE_RUN_ID, "name": name, "profile": profile}, sort_keys=True).encode()).hexdigest()[:12]
    return RESULT_ROOT / f"{name}_{signature}.json"

def evaluate_frozen(model, tokenizer, name, profile, weight_metrics):
    output_path = trial_path(name, profile)
    if output_path.exists() and not FORCE_RERUN:
        with output_path.open() as handle:
            payload = json.load(handle)
        print(f"Reusing {output_path.name}")
    else:
        config = KVCacheEvalConfig(**profile)
        started = time.perf_counter()
        metrics = evaluate_kv_cache(model, batches_for(tokenizer, config), config, "cuda")
        metrics["seconds"] = time.perf_counter() - started
        payload = {"trial": name, "model": MODEL_ID, "git_sha": commit, "source_run": SOURCE_RUN_ID, "kv_config": asdict(config), "weight_metrics": weight_metrics, "metrics": metrics, "environment": environment_record()}
        with output_path.open("w") as handle:
            json.dump(payload, handle, indent=2)
        print(f"Wrote {output_path.name}: KL={metrics['mean_teacher_kl']:.6f}, bpv={metrics['effective_kv_bpv']:.3f}")
    trial_records[name] = payload
    return payload

def load_model():
    set_seed(0)
    model, tokenizer, _ = load_hf_model(MODEL_ID, torch.float16, "cuda", "multimodal_lm")
    model.eval()
    return model, tokenizer

def patch_weights(model):
    quant = QuantConfig(bits=4, codebook="gaussian", scale="mse_search", group_size=128, error_comp="none", seed=0)
    patch = PatchConfig(quant=quant, rotation="fwht", block=128, mode="consistent", fallback=True, seed=0, include=["model.language_model.layers."], exclude=["linear_attn.in_proj_a", "linear_attn.in_proj_b"])
    started = time.perf_counter()
    patch_model(model, patch)
    metrics = footprint_metrics(model, {})
    metrics["patch_seconds"] = time.perf_counter() - started
    return metrics

### 6. Confirm and run six frozen evaluations

Set `CONFIRM_FROZEN_TRANSFER=True` in the parameter cell. This loads and patches the model once, then evaluates three maps on two cached context datasets. It does not repeat dynamic candidate scoring.

In [ ]:
assert CONFIRM_FROZEN_TRANSFER, "Set CONFIRM_FROZEN_TRANSFER=True before running transfer trials."
model, tokenizer = load_model()
weight_metrics = patch_weights(model)
profiles = {
    "short_map__short_ctx": frozen_profile(short_dynamic, short_recipe),
    "short_map__long_ctx": frozen_profile(long_dynamic, short_recipe),
    "long_map__short_ctx": frozen_profile(short_dynamic, long_recipe),
    "long_map__long_ctx": frozen_profile(long_dynamic, long_recipe),
    "mixed_map__short_ctx": frozen_profile(short_dynamic, mixed_recipe),
    "mixed_map__long_ctx": frozen_profile(long_dynamic, mixed_recipe),
}
for name, profile in profiles.items():
    evaluate_frozen(model, tokenizer, name, profile, weight_metrics)
model = None
gc.collect()
torch.cuda.empty_cache()

## Checks

### 7. Enforce replay equivalence before interpreting transfer

In [ ]:
import math

def replay_check(original, replay):
    expected = original["metrics"]
    actual = replay["metrics"]
    return {
        "kl_expected": expected["mean_teacher_kl"],
        "kl_actual": actual["mean_teacher_kl"],
        "bpv_expected": expected["effective_kv_bpv"],
        "bpv_actual": actual["effective_kv_bpv"],
        "kl_relative_drift": kl_relative_drift,
        "cosine_absolute_drift": cosine_absolute_drift,
        "top1_absolute_drift": top1_absolute_drift,
        "nll_absolute_drift": nll_absolute_drift,
        "kl_match": math.isclose(actual["mean_teacher_kl"], expected["mean_teacher_kl"], rel_tol=REPLAY_KL_RTOL, abs_tol=REPLAY_KL_ATOL),
        "cosine_match": cosine_absolute_drift <= REPLAY_COSINE_ATOL,
        "top1_match": top1_absolute_drift <= REPLAY_TOP1_ATOL,
        "bpv_match": math.isclose(actual["effective_kv_bpv"], expected["effective_kv_bpv"], rel_tol=0, abs_tol=1e-9),
    }
    result["replay_pass"] = all(result[name] for name in ("kl_match", "cosine_match", "top1_match", "bpv_match"))
    return result

replay_checks = {
    "short": replay_check(short_dynamic, trial_records["short_map__short_ctx"]),
    "long": replay_check(long_dynamic, trial_records["long_map__long_ctx"]),
}
print(json.dumps(replay_checks, indent=2))
assert all(check["replay_pass"] for check in replay_checks.values()), "Frozen replay failed; do not interpret transfer results."

## Results

### 8. Compare fixed maps on both contexts

In [ ]:
import pandas as pd

def result_row(map_name, context, payload):
    metrics = payload["metrics"]
    return {
        "map": map_name, "context": context,
        "teacher_kl": metrics["mean_teacher_kl"],
        "logit_cosine": metrics["mean_logit_cosine"],
        "top1": metrics["top1_agreement"],
        "nll_delta": metrics["nll_delta"],
        "effective_bpv": metrics["effective_kv_bpv"],
        "packed_kv_MB": metrics["packed_kv_bytes"] / 1e6,
        "compression": metrics["kv_compression_ratio"],
    }

rows = []
for map_name in ("short", "long", "mixed"):
    rows.append(result_row(map_name, "short_256", trial_records[f"{map_name}_map__short_ctx"]))
    rows.append(result_row(map_name, "long_1024", trial_records[f"{map_name}_map__long_ctx"]))
rows.extend([
    result_row("uniform_k3_v3", "short_256", short_uniform3),
    result_row("uniform_k3_v3", "long_1024", long_uniform3),
    result_row("uniform_k4_v4", "long_1024", long_uniform4),
])
results = pd.DataFrame(rows)
display(results.sort_values(["context", "teacher_kl"]).style.format({"teacher_kl": "{:.6f}", "logit_cosine": "{:.6f}", "top1": "{:.4f}", "nll_delta": "{:+.4f}", "effective_bpv": "{:.3f}", "packed_kv_MB": "{:.3f}", "compression": "{:.2f}x"}))

In [ ]:
uniform3 = results[results["map"] == "uniform_k3_v3"].set_index("context")
context_optimum = {
    "short_256": float(trial_records["short_map__short_ctx"]["metrics"]["mean_teacher_kl"]),
    "long_1024": float(trial_records["long_map__long_ctx"]["metrics"]["mean_teacher_kl"]),
}
map_decisions = []
for map_name in ("short", "long", "mixed"):
    candidate = results[results["map"] == map_name].set_index("context")
    beats_uniform = all(candidate.loc[context, "teacher_kl"] < uniform3.loc[context, "teacher_kl"] and candidate.loc[context, "effective_bpv"] <= uniform3.loc[context, "effective_bpv"] + 1e-9 for context in context_optimum)
    regrets = [candidate.loc[context, "teacher_kl"] / optimum - 1 for context, optimum in context_optimum.items()]
    map_decisions.append({"map": map_name, "beats_uniform_both": beats_uniform, "worst_regret_vs_context_optimum": max(regrets), "mean_regret_vs_context_optimum": sum(regrets) / len(regrets)})
decision_frame = pd.DataFrame(map_decisions).sort_values(["beats_uniform_both", "worst_regret_vs_context_optimum"], ascending=[False, True])
eligible = decision_frame[decision_frame["beats_uniform_both"]]
recommendation = eligible.iloc[0]["map"] if len(eligible) else "context_bucketed"
display(decision_frame.style.format({"worst_regret_vs_context_optimum": "{:+.2%}", "mean_regret_vs_context_optimum": "{:+.2%}"}))
print({"recommendation": recommendation})

In [ ]:
import matplotlib.pyplot as plt

plot_frame = results[results["map"].isin(["short", "long", "mixed", "uniform_k3_v3"])].pivot(index="context", columns="map", values="teacher_kl").loc[["short_256", "long_1024"]]
ax = plot_frame.plot(kind="bar", figsize=(10, 5), rot=0)
ax.set_ylabel("Mean teacher KL (lower is better)")
ax.set_xlabel("Held-out context")
ax.set_title("Frozen RotQuant K/V map transfer across context lengths")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plot_path = RESULT_ROOT / "kv_frozen_transfer.png"
plt.savefig(plot_path, dpi=180)
plt.show()
print(f"Wrote {plot_path}")

### 9. Persist the decision record

In [ ]:
results_path = RESULT_ROOT / "kv_frozen_transfer.csv"
results.to_csv(results_path, index=False)
summary = {
    "git_sha": commit, "source_run_id": SOURCE_RUN_ID,
    "model": MODEL_ID, "replay_checks": replay_checks,
    "mixed_allocator": mixed_allocator,
    "short_recipe": short_recipe, "long_recipe": long_recipe,
    "mixed_recipe": mixed_recipe,
    "map_decisions": map_decisions, "recommendation": recommendation,
}
summary_path = RESULT_ROOT / "kv_frozen_transfer_summary.json"
with summary_path.open("w") as handle:
    json.dump(summary, handle, indent=2)
ledger_path = RESULT_ROOT / "experiment_log_entry.md"
ledger_path.write_text(f"""## Frozen Qwen3.5-4B K/V transfer\n\n- Git SHA: `{commit}`\n- Source matrix: `{SOURCE_RUN_ID}`\n- Replay gates: `{json.dumps(replay_checks)}`\n- Recommendation: `{recommendation}`\n- Raw table: `kv_frozen_transfer.csv`\n- Plot: `kv_frozen_transfer.png`\n\nA universal map is recommended only when it beats uniform K3/V3 at no more exact bytes on both held-out contexts.\n""")
print(json.dumps({"recommendation": recommendation, "results": str(results_path), "summary": str(summary_path)}, indent=2))

### 10. Archive the compact handoff

In [ ]:
import shutil

archive_path = Path(shutil.make_archive("/content/qwen35_kv_frozen_transfer", "zip", RESULT_ROOT))
print(f"Created {archive_path} ({archive_path.stat().st_size / 1e6:.2f} MB)")
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(archive_path))

## Takeaways

Send back `kv_frozen_transfer.csv`, `kv_frozen_transfer_summary.json`, and the plot. If one fixed map passes both contexts, promote it into the joint weight-plus-cache notebook. If none passes, use the context-bucketed short/long maps and measure the operational cost of map switching.